# Calcite–dolomite 1-D advection: the column as parts of a `Material`

You give the column as measured: a pore solution and the solid as `Part`s, each a `Material`. xGEM2MT scales them to the volume and the porosity of every node, so you do not calculate node amounts yourself.

For the amounts of one node given directly, see [calcite-dolomite-1d-advection-material.ipynb](calcite-dolomite-1d-advection-material.ipynb).

## How the parts work

The GEMS3K files in `CalciteColumn/` define the chemical system. After `initNodes()`, the inflowing fluid is set with `composition=` and the column with:
* `pore_solution`: the pore solution, any amount. It is the reference of each node (by default the node gets the water of the solution as given; `water_mass` scales it to another mass of water; `water_saturation` gives its share of the pore volume, default 1).
* `parts`: the other phases and solids, each a `Material`, the volume that it fills (`of="pore"` for a gas or another liquid, `of="solid"` for a solid) and its share: `Part(gas, fraction=0.1, of="pore")`, `Part(quartz, fraction=0.95, of="solid")` (the solid fractions add up to 1), or `Part(organic, mass=0.005, of="solid")` in kg (its volume is taken from the solid volume).

xGEM2MT calculates the amounts of each node: pore volume = volume of the pore water / water saturation, solid volume = pore volume × (1 − porosity) / porosity (the transport porosity of the node). The volume of each part comes from a GEM calculation at the T and P of the node (the models of the chemical system, e.g. a non-ideal gas), or from a `density` that you give. A gas that dissolves in part is iterated, so that it fills its share. Thus the porosity of the chemical system is equal to the transport porosity.

Below, the column has one solid part (the calcite and the inert solid). The amounts are the same as in the nodes notebook, except for the difference between the GEM volume and the volume in the files (less than 0.01 %).


## Make the materials

Only the **ratios** matter here: xGEM2MT scales the pore solution and the solid to the node (see above). The materials are made with the xGEMS `Material` class. Each pore solution gets a little O$_2$, so that the redox state is defined.

**The inert solid.** The chemical system has an element `Inert` (a solid phase `Inert` that does not react; in the original xGEMS benchmark files it is tin, Sn). It stands for the solid matrix of the rock. Mode 'A' does not need it for the transport: it moves only the dissolved amounts.

**Porosity.** The transport porosity is a number that you give (`TransportProperties(porosity=...)` or `setInitialNodeEffectivePorosity`). With parts, xGEM2MT sets the solid volume so that the porosity of the chemical system (water volume / node volume) equals it. Here it is 0.32.

In [ ]:
from xgems import ChemicalEngine, Material

# The xGEMS engine gives Material the element list and the molar masses
gems = ChemicalEngine("CalciteColumn/CalciteColumn-dat.lst")

fluid = Material(gems, "MgCl2 solution")       # inflowing fluid, system 0: amounts of one node
fluid.add("H2O", 1.0, "kg")
fluid.add("MgCl2", 1e-3, "mol")                # 1 mmol per kg of water
fluid.add("O2", 1e-5, "mol")
fluid.add({"Inert": 100.0}, 1.0, "mol")        # inert solid matrix

pore_solution = Material(gems, "pore solution")  # column pore water, any amount (here 1 kg)
pore_solution.add("H2O", 1.0, "kg")
pore_solution.add("O2", 1e-4, "mol")

dry_solid = Material(gems, "dry solid")        # calcite and inert solid, any amount: it is scaled
dry_solid.add("CaCO3", 3.405e-4, "mol")
dry_solid.add({"Inert": 100.0}, 1.0, "mol")

In [ ]:
fluid

### From a column experiment to the model

| Experiment | Model | How |
|---|---|---|
| column length L | column length | `setSpatialDimensions(L)`; node length L/n not more than 2 × dispersivity |
| flow rate Q, cross-section area A, porosity φ | velocity | v = Q / (A · φ) → `setFluidAdvectionVelocity(v)` |
| porosity φ (water in the saturated column / column volume) | porosity | `setInitialNodeEffectivePorosity(φ)` |
| tracer breakthrough curve (Cl⁻, Br⁻) | dispersivity | fit the curve → `setInitialDispersivity` |
| duration t | time | `setTau(0, t, step)`; pore volumes = v·t/L |
| inflowing solution (mol/kg water) | composition of nodes 0 and 1 | 1 kg of water + dissolved amounts per kg of water |
| initial pore water | composition of the column nodes | 1 kg of water + dissolved amounts per kg of water |
| minerals packed in the column | solid part of the column nodes | `Part(solid, fraction=1.0, of="solid")`; xGEM2MT scales it to the pore volume and the porosity |
| effluent | node n − 2 | the last node keeps its start composition (outlet) |

What each node needs:
* **Nodes 0 and 1 (inflow):** only the composition of the inflowing solution. Their transport values are not used.
* **Column nodes:** the pore solution and the solid parts (as measured), and the transport values. xGEM2MT scales them to the node.

The porosity is used twice: in the dispersion coefficient (diffusion part only), and to scale the solid to the pore volume. More: section 8 of [MassTransport in Python](https://github.com/gemshub/xgem2mt/blob/master/docs/python-mass-transport.md).

## Set up the task

The same as in the original notebook.

All functions and their parameters: [MassTransport in Python](https://github.com/gemshub/xgem2mt/blob/master/docs/python-mass-transport.md).

### Names of the transport values

Some names look alike but are different values. This notebook uses only the names in the first column:

| Name | Symbol | Unit | Meaning | All nodes (before `initNodes`) | Some nodes (`TransportProperties`) |
|---|---|---|---|---|---|
| velocity | v | m/s | pore velocity of the water | `setFluidAdvectionVelocity` | `velocity` (changes only the dispersion) |
| porosity | φ | – | pore volume / bulk volume | `setInitialNodeEffectivePorosity` | `porosity` |
| **dispersivity** | α | m | a length of the porous medium: how much the flow spreads the solutes | `setInitialDispersivity` | `dispersivity` |
| **diffusivity** | D<sub>w</sub> | m²/s | diffusion coefficient of the solutes in water | `setInitialDiffusivity` | `diffusivity` |
| tortuosity | τ | – | how much longer the diffusion path is than the column | `setInitialTortuosityFactor` | `tortuosity` |
| **dispersion coefficient** | D | m²/s | the program calculates it: D = α·v + D<sub>w</sub>·φ/τ | – | – |
| node length | Δx | m | column length / number of nodes | `setSpatialDimensions`, `MassTransport(mode, n)` | – |
| time step | Δt | s | the program calculates it (see the comment at `setTau`) | – | – |

Here the diffusivity is 0, so D = α·v = 0.0067 m × 9.375·10⁻⁶ m/s = 6.3·10⁻⁸ m²/s. Porosity, diffusivity and tortuosity change only the diffusion part of D.

Not used in mode 'A', thus not set here: `setInitialTotalNodeVolume`, `setInitialEffectivePermeability`, `setTypeFluxPhase`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from xgem2mt import MassTransport, TransportProperties

In [ ]:
results = {}

def monitor(engine, step, time):
    nodes = range(engine.nNodes())
    aq = [engine.elementAmountsInPhase(n, "aq_gen") / engine.phaseMass(n, "aq_gen") for n in nodes]
    results[time] = {
        "Ca": [c[engine.indexElement("Ca")] for c in aq],
        "Mg": [c[engine.indexElement("Mg")] for c in aq],
        "Cl": [c[engine.indexElement("Cl")] for c in aq],
        "Calcite":  [engine.phaseAmount(n, "Calcite") for n in nodes],
        "Dolomite": [engine.phaseAmount(n, "Dolomite-dis") for n in nodes],
    }

In [ ]:
nnodes = 80

# (1) Create the MassTransport simulator in 'A' mode (1-D advection, finite differences)
mt_task = MassTransport("A", nnodes)
mt_task.setName("Calcite-dolomite 1-D advection benchmark")

# (2) Time: start, end, step, in s. The program calculates the time step itself, as the smaller of
#       0.5 * node length / velocity                       advection limit:  333 s here
#       node length**2 / (4 * dispersion coefficient)      dispersion limit: 155 s here
#     (node length 0.00625 m, dispersion coefficient 6.3e-8 m2/s: see the table of names above).
#     The run stops at the first step at or after 'end' (here 21144 s), or after (end - start)/step + 1
#     steps. Keep 'step' smaller than the time step, or the run stops too early.
mt_task.setTau(0.0, 21000.0, 10.0)
mt_task.setSpatialDimensions(0.5)            # column length, m; node length = column length / nnodes (y, z: not used)
mt_task.setSIA("+")  # GEM start in each node: "+" from the previous step (SIA, ~5x faster), "-" from scratch (AIA)
mt_task.setOutput(False)

# Transport values for all nodes (names: see the table above)
mt_task.setTimeStepReductionFactor(1.0)          # the time step is divided by this factor; 1 = not reduced
mt_task.setFluidAdvectionVelocity(9.375e-6)      # velocity, m/s (pore velocity)
mt_task.setInitialNodeEffectivePorosity(0.32)    # porosity
mt_task.setInitialDispersivity(0.0067)           # dispersivity, m
mt_task.setInitialDiffusivity(0.0)               # diffusivity, m2/s (0: no diffusion)
mt_task.setInitialTortuosityFactor(1.0)          # tortuosity
mt_task.setMonitoring(monitor)

# (3) Read the GEMS3K files. They define the chemical system; the compositions are replaced below
mt_task.initNodes("CalciteColumn/CalciteColumn-dat.lst")

## Set up the nodes

**Which nodes are fluid and which are column?** Each node is set up from one *initial system*. The initial systems are the dbr files in `CalciteColumn-dat.lst`, in their order: system 0 = `CalciteColumn-dbr-0-0000.dat`, system 1 = `CalciteColumn-dbr-0-0001.dat`. In mode 'A' xGEM2MT gives system 0 to nodes 0 and 1 and system 1 to all other nodes (`setDistributing(node, system)` before `initNodes` changes this). Thus system 0 is the inflowing fluid and system 1 is the column. Below, `setSystem` replaces the compositions of these two groups.

The **node type** is a different thing: it is the boundary condition (3 = source for nodes 0 and 1, −3 = outlet for the last node, 0 = normal). Mode 'A' does not use it: nodes 0 and 1 always stay constant and the last node is the outlet.

In [ ]:
print("system 0:", mt_task.nodesOfSystem(0))
print("system 1:", mt_task.nodesOfSystem(1)[:3], "...", mt_task.nodesOfSystem(1)[-1])

from xgem2mt import Part

mt_task.setSystem(0, composition=fluid)    # setSystem calls fluid.b_dict() itself
# The column: the pore solution (1 kg of water) and one solid part. xGEM2MT scales the solid to the pore
# volume and the transport porosity of each node. Here the solid is one part, so its fraction is 1.
mt_task.setSystem(1, pore_solution=pore_solution,
                  parts=[Part(dry_solid, fraction=1.0, of="solid")])

engine = mt_task.getChemicalEngineArray()
print(f"node 0: pH {engine.pH(0):.2f};  node 5: pH {engine.pH(5):.2f}, calcite {engine.phaseAmount(5, 'Calcite'):.3e} mol")
print("node 5:", mt_task.transportProperties(5))
print("chemical porosity of node 5: %.4f" % (engine.phaseVolume(5, "aq_gen") / engine.systemVolume(5)))

`print(mt_task)` (or `mt_task.summary()`) shows the setup: the column, the time step that the program uses, which nodes are fluid and column, their properties and phases, and which function changes each value.

In [ ]:
print(mt_task)

**How much is in a node?** xGEM2MT scaled each column node: 1 kg of water (1.00 L, the water of the pore solution as given) and the solid with the volume pore volume × (1 − porosity) / porosity = 2.13 L. At each time step, advection moves the fraction velocity × time step / node length (here 0.23) of the dissolved amounts of a node to the next node.

In [ ]:
for n in (0, 1, 2, nnodes - 1):
    print(f"node {n:2d}: system {mt_task.nodeSystem(n)}, type {mt_task.nodeType(n):2d}, "
          f"mass {engine.systemMass(n):.2f} kg, volume {engine.systemVolume(n)*1e3:.2f} L, "
          f"water {engine.phaseVolume(n, 'aq_gen')*1e3:.2f} L, inert {engine.phaseVolume(n, 'Inert')*1e3:.2f} L, "
          f"calcite {engine.phaseAmount(n, 'Calcite'):.2e} mol")

Single nodes or lists of nodes are set up the same way, for example `mt_task.setNodes(range(60, 80), transport=TransportProperties(porosity=0.2))`. `composition` can also be a dict `{element: mol}` or a list of element amounts.

## Run the simulation

In [ ]:
import time

start = time.perf_counter()
mt_task.runMassTransport()
elapsed = time.perf_counter() - start

times = sorted(results)
x = (np.arange(nnodes) + 0.5) * 0.5 / nnodes   # node centres, m
end = results[times[-1]]
print(f"Simulated: {len(times)} time steps up to t = {times[-1]:.0f} s of model time")
print(f"Computing time: {elapsed:.1f} s")

## Plot the results

Dissolved Ca, Mg and Cl along the column at the final time:

In [ ]:
for name in ("Ca", "Mg", "Cl"):
    plt.plot(x, end[name], label=name)
plt.xlabel("Distance, m"); plt.ylabel("Concentration, mol/kg"); plt.legend(); plt.show()

Calcite and dolomite at the final time:

In [ ]:
plt.plot(x, end["Calcite"], label="Calcite")
plt.plot(x, end["Dolomite"], label="Dolomite")
plt.xlabel("Distance, m"); plt.ylabel("Amount, mol"); plt.legend(); plt.show()

The dolomite zone moves along the column with time:

In [ ]:
for t in times[::27]:
    plt.plot(x, results[t]["Dolomite"], label=f"{t:.0f} s")
plt.xlabel("Distance, m"); plt.ylabel("Dolomite, mol"); plt.legend(); plt.show()

**Breakthrough curves**: the dissolved amounts against time at fixed points of the column, as in the effluent of a column experiment. The x axis is the number of pore volumes, velocity × time / column length.

This run lasts 0.39 pore volumes, so the reaction front (at about 0.17 m) does not get to the outlet. The plot therefore shows three points inside the column and the outlet. The outlet is node n − 2: the last node keeps its start composition. To compare with the effluent of an experiment, run for at least 1 to 2 pore volumes, e.g. `setTau(0.0, 2 * 0.5 / 9.375e-6, 10.0)` (107 000 s).

In [ ]:
velocity, length = 9.375e-6, 0.5                  # as in the setup above
pore_volumes = np.array(times) * velocity / length
points = {"0.05 m": 8, "0.10 m": 16, "0.15 m": 24, "outlet": nnodes - 2}   # node = position / node length

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, node in points.items():
    axes[0].plot(pore_volumes, [results[t]["Mg"][node] for t in times], label=label)
    axes[1].plot(pore_volumes, [results[t]["Ca"][node] for t in times], label=label)
axes[0].set(xlabel="Pore volumes", ylabel="Mg, mol/kg", title="Mg (from the inflow)")
axes[1].set(xlabel="Pore volumes", ylabel="Ca, mol/kg", title="Ca (from calcite)")
axes[0].legend(); axes[1].legend(); plt.tight_layout(); plt.show()